In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [12]:
DATA_FOLDER = "preprocessed_avoiding"

# Select only relevant trials from whole data set

In [13]:
class Condition(pd.DataFrame):

    def __init__(self, file: str, condition:str):
        super().__init__(pd.read_csv(file))
        self.condition = condition

    def get_condition(self):
        return self.condition

In [15]:
ai = Condition(f"{DATA_FOLDER}/ai.csv", "ai")
ad = Condition(f"{DATA_FOLDER}/ad.csv", "ad")

In [16]:
def get_relevant_row(df: pd.DataFrame):
    """Gets the actual row per trial"""
    type = df.loc[:, "task"].iloc[0]
    if type == "avoiding":
        switched = df.loc[df["switched_side"] == 1]
        return switched.iloc[0] if len(switched) > 0 else df.iloc[-1]
    elif type == "reaching":
        return df.iloc[-1]
    else:
        raise ValueError(f"Faulty trial type found! No typ of {type}")

def find_relevant_entry(df: pd.DataFrame):
    """Separates the phase df into blocks and returns one row per block"""
    #print(df)
    return df.groupby(["trial_index"]).apply(get_relevant_row, include_groups=False)

In [17]:
for cond in [ai, ad]:
    nested_group = ai.groupby(["participant_id", "phase", "block"])
    grouped = nested_group.apply(find_relevant_entry, include_groups=False)
    grouped.to_csv(f"{DATA_FOLDER}/{cond.get_condition()}_single_trial.csv")

In [18]:
del ai
del ad

# Calculate mean and variance per block

In [20]:
ai = Condition(f"{DATA_FOLDER}/ai_single_trial.csv", "ai")
ad = Condition(f"{DATA_FOLDER}/ad_single_trial.csv", "ad")

In [21]:
def get_mean_variance_median_per_block(df: pd.DataFrame):
    df["difference"] = df.loc[:,"target_pos_y"] - df.loc[:,"current_pos_y"]
    return pd.DataFrame({
        "diff_mean": [df["difference"].mean()],
        "diff_variance": [df["difference"].var()]
    })

In [24]:
for cond in [ai, ad]:
    grouped = cond.groupby(["participant_id", "phase", "block", "task"]).apply(get_mean_variance_median_per_block, include_groups=False)
    grouped.to_csv(f"{DATA_FOLDER}/{cond.get_condition()}_single_trial_measures.csv")

In [25]:
grouped

diff_mean  diff_variance
participant_id phase     block task                                
12ai           Post-Test 1     avoiding 0   2.238482      83.297339
                               reaching 0  -0.981089      85.870513
               Pre-Test  1     avoiding 0  -2.520732     220.395017
                               reaching 0  -5.231571     226.201514
               Recap     1     avoiding 0   1.391675     106.209704
...                                              ...            ...
8ai            Training  1     avoiding 0  -0.290475      23.587322
                         2     avoiding 0   0.745350      32.046748
                         3     avoiding 0   0.371050      26.260145
                         4     avoiding 0   0.280450      32.023132
                         5     avoiding 0   0.078675      27.638584

[90 rows x 2 columns]